<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 4 · Seq2seq and attention

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/04-seq2seq-attention.ipynb)

**From Traditional NLP to Modern LLMs** · Day 1 · 95 minutes (45 briefing, 50 lab)

Encoder-decoder models, the fixed-vector bottleneck, and attention as the fix.

**By the end of this notebook you can:**

- Assemble an encoder-decoder's teacher-forced forward pass and loss
- Explain the fixed-vector bottleneck and measure it by input length
- Implement dot-product and additive attention and read attention maps against the expected alignment

**Stack:** PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "04-seq2seq-attention"
_CONTENT_SHA = "8a83f4e6e2abb753"
_EXERCISES = {1: ('forward_plain', 'seq2seq_loss'), 2: ('exact_match',), 3: ('masked_attention', 'dot_product_attention'), 4: ('additive_score',), 5: ('alignment_hit_rate',), 6: ('beam_search',)}
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Fix TODO {n}, or go on"
                f" with workshop.use_reference({n!r})."
            )

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

**Use a GPU runtime:** *Runtime → Change runtime type → T4 GPU*. The lab also runs on a CPU, but the two training runs then take much longer. On a CPU runtime, set `QUICK = True` in the next cell: each model then trains for an eighth of the steps, and the accuracy checkpoints, whose targets assume the full budget, are skipped. No API key and no installs are needed: PyTorch, NumPy and matplotlib are preinstalled on Colab.

**How the lab works.** There are five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one or two short functions;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

**Run time.** Not yet timed on a Colab T4. On the 4-core CPU the lab was built on, shared with other jobs, Run all took 107 minutes at full settings (81 minutes for the model without attention, 24 for the model with it) and 5 to 7 minutes with `QUICK = True`, depending on the load. A T4 is expected to be much faster, but that is an expectation, not a measurement.

**Two training runs.** The model without attention trains while you work on Exercise 2, and the model with attention trains while you work on Exercise 4. Start each training cell, then keep reading.

**Notation** follows the briefing: `h` are the encoder states $h_i$, `s` the decoder states $s_t$, `alpha` the attention weights $\alpha_{t,i}$, and `h_bar` the context vector $\bar h_t$ (not `c`, which was the LSTM cell state in Lab 3).

In [ ]:
# Setup: imports, seeds, device and run settings. Everything here is preinstalled on Colab.
import datetime as dt
import json
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run settings. The model without attention and the model with attention use exactly
# these: the same sizes, the same number of steps, the same optimizer and the same batches.
D_EMB = 32  # embedding size d
D_H = 128  # hidden size d_h of the encoder and of the decoder
BATCH = 128  # sequence pairs per step
LR = 2e-3  # Adam learning rate
CLIP = 1.0  # gradient-norm clipping threshold (Lab 3)
# QUICK = True trains each model for an eighth of the steps: for a CPU runtime or a dry run.
# The accuracy checkpoints are then skipped, because the targets assume the full budget.
QUICK = os.environ.get("NLP_LLMS_QUICK") == "1"
STEPS_FULL = 4000  # training steps per model
STEPS = STEPS_FULL // 8 if QUICK else STEPS_FULL

NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
LAB_START = time.time()
print(f"PyTorch {torch.__version__} · device {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"d = {D_EMB}, d_h = {D_H}, batch = {BATCH}, Adam lr = {LR}, {STEPS} steps per model")
if QUICK:
    print(f"\nQUICK = True: {STEPS} steps per model; the accuracy checkpoints will be skipped.")
if device.type == "cpu" and not QUICK:
    print("\nNo GPU found. The lab works on a CPU, but each training run takes several times "
          "longer than on a T4. Switch to a T4 runtime, or set QUICK = True above.")

## Data: dates to ISO format

The task is character-level **transduction**: read dates written for people and write them in ISO format, `YYYY-MM-DD`.

| Source | Target |
|---|---|
| `3 March 2021` | `2021-03-03` |
| `from Jul 4, 1999 to 2001/12/25` | `1999-07-04 2001-12-25` |

**How the input length grows.** A source holds $K$ dates, $K = 1$ to $4$, joined by short connectives (`from … to …`, `between … and …`, `and`, `,`, `;`). The target is the $K$ ISO dates in the same order, separated by a space. So the amount the decoder must remember grows with the input. This is what a fixed-size vector cannot absorb (briefing, Section 3).

The generator is in the next cell and is fully deterministic:

- Each date is drawn uniformly between 1 January 1900 and 31 December 2099.
- Each date is written in one of six formats, chosen uniformly: `3 March 2021`, `March 3, 2021`, `3 Mar 2021`, `Mar 3, 2021`, `3rd March 2021`, `2021/3/3`. All are unambiguous: the month is a word, or the numerals are year first. There is no `03/04/21`.
- The **test set** has 500 sources for each $K$ (2,000 in all), drawn with `random.Random(0)`. The **training set** has 25,000 sources for each $K$, drawn with `random.Random(1)`; any source that is also in the test set is skipped. No source appears twice within a split.
- Nothing is downloaded. Every participant gets the same examples.

All metrics in this lab are reported per **bucket** $K = 1, 2, 3, 4$. No setting is tuned in this lab, so there is no validation split: the settings in the setup cell are fixed in advance, and each model is scored on the test set once.

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
FIRST_DAY, LAST_DAY = dt.date(1900, 1, 1).toordinal(), dt.date(2099, 12, 31).toordinal()


def ordinal(n):
    """1 -> '1st', 2 -> '2nd', 11 -> '11th', 23 -> '23rd'."""
    suffix = "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


FORMATS = [  # six unambiguous surface forms of one date
    lambda d: f"{d.day} {MONTHS[d.month - 1]} {d.year}",  # 3 March 2021
    lambda d: f"{MONTHS[d.month - 1]} {d.day}, {d.year}",  # March 3, 2021
    lambda d: f"{d.day} {MONTHS[d.month - 1][:3]} {d.year}",  # 3 Mar 2021
    lambda d: f"{MONTHS[d.month - 1][:3]} {d.day}, {d.year}",  # Mar 3, 2021
    lambda d: f"{ordinal(d.day)} {MONTHS[d.month - 1]} {d.year}",  # 3rd March 2021
    lambda d: f"{d.year}/{d.month}/{d.day}",  # 2021/3/3
]
PAIR_CONNECTIVES = [("from ", " to "), ("between ", " and "), ("", " and "), ("", "; ")]


def make_example(rng, K):
    """One source with K dates. Returns {"K", "src", "tgt", "spans"}.

    spans[k] = (start, end) is the character span of date k in the source, end exclusive.
    """
    dates = [dt.date.fromordinal(rng.randint(FIRST_DAY, LAST_DAY)) for _ in range(K)]
    texts = [rng.choice(FORMATS)(d) for d in dates]
    prefix, joins = "", []
    if K == 2:
        prefix, middle = rng.choice(PAIR_CONNECTIVES)
        joins = [middle]
    elif K > 2:
        joins = [", "] * (K - 2) + [" and "] if rng.random() < 0.5 else ["; "] * (K - 1)
    src, spans = prefix, []
    for k, text in enumerate(texts):
        spans.append((len(src), len(src) + len(text)))
        src += text + (joins[k] if k < K - 1 else "")
    tgt = " ".join(d.isoformat() for d in dates)
    return {"K": K, "src": src, "tgt": tgt, "spans": spans}


def make_split(seed, n_per_k, exclude=frozenset()):
    """n_per_k distinct sources for each K = 1..4, none of them in `exclude`."""
    rng, examples = random.Random(seed), []
    for K in (1, 2, 3, 4):
        seen = set()
        while len(seen) < n_per_k:
            ex = make_example(rng, K)
            if ex["src"] in exclude or ex["src"] in seen:
                continue
            seen.add(ex["src"])
            examples.append(ex)
    return examples

In [ ]:
started = time.time()
# The briefing's example, used for the heat-map in Exercise 5. It is kept out of training.
SHOWCASE = {"K": 1, "src": "3 March 2021", "tgt": "2021-03-03", "spans": [(0, 12)]}
test_set = make_split(seed=0, n_per_k=500)
train_set = make_split(seed=1, n_per_k=25_000,
                       exclude=frozenset(ex["src"] for ex in test_set) | {SHOWCASE["src"]})
assert not {ex["src"] for ex in train_set} & {ex["src"] for ex in test_set}
print(f"{len(train_set):,} training and {len(test_set):,} test examples "
      f"({time.time() - started:.1f} s)\n")

print(" K  test sources  source characters (mean / max)  target characters")
for K in (1, 2, 3, 4):
    bucket = [ex for ex in test_set if ex["K"] == K]
    lengths = [len(ex["src"]) for ex in bucket]
    print(f"{K:>2}  {len(bucket):>12}  {np.mean(lengths):>22.1f} / {max(lengths):<3}"
          f"  {len(bucket[0]['tgt']):>17}")

print()
for K in (1, 2, 3, 4):
    ex = next(ex for ex in test_set if ex["K"] == K)
    print(f"K = {K}  {ex['src']!r}\n       -> {ex['tgt']!r}")

**Exercise 0 · Read the data (3 minutes).** Look at the printed examples. Which parts of the target can be copied character by character, and which need a lookup (for example `Mar` → `03`)? Which part of the source must the decoder *find* when it writes the second date?

**Tokens and batches (provided).** Tokens are characters. The source and the target have separate vocabularies. The target vocabulary $V$ has 15 entries: the ten digits, `-`, the space, and three special tokens. `<pad>` fills the end of shorter sequences in a batch, `<bos>` is the start token $y_0$ fed to the decoder at the first step, and `<eos>` ends every target.

In [ ]:
PAD, BOS, EOS = 0, 1, 2
SRC_CHARS = sorted(set("".join(MONTHS) + "0123456789 ,;/" + "stndrh" + "frombetweenandto"))
SRC_ITOS = ["<pad>"] + SRC_CHARS  # id -> source character
SRC_STOI = {ch: i for i, ch in enumerate(SRC_ITOS)}
TGT_ITOS = ["<pad>", "<bos>", "<eos>"] + list("0123456789- ")  # id -> target token
TGT_STOI = {ch: i for i, ch in enumerate(TGT_ITOS)}
assert set("".join(ex["src"] for ex in train_set + test_set)) <= set(SRC_CHARS)
print(f"Source vocabulary: {len(SRC_ITOS)} entries · target vocabulary |V| = {len(TGT_ITOS)}")


def encode_batch(examples):
    """Pad a list of examples into tensors.

    src: (B, S) source ids, 0-padded.  src_len: (B,) true source lengths.
    tgt: (B, T + 2) target ids: <bos>, the T target characters, <eos>, then 0-padding.
    """
    S = max(len(ex["src"]) for ex in examples)
    T2 = max(len(ex["tgt"]) for ex in examples) + 2
    src = [[SRC_STOI[ch] for ch in ex["src"]] + [PAD] * (S - len(ex["src"])) for ex in examples]
    tgt = [[BOS] + [TGT_STOI[ch] for ch in ex["tgt"]] + [EOS] + [PAD] * (T2 - 2 - len(ex["tgt"]))
           for ex in examples]
    src_len = [len(ex["src"]) for ex in examples]
    return torch.tensor(src), torch.tensor(src_len), torch.tensor(tgt)


def ids_to_string(ids):
    """Target ids -> string, stopping at the first <eos>."""
    out = []
    for i in ids:
        if i == EOS:
            break
        out.append(TGT_ITOS[i] if i > EOS else "")
    return "".join(out)


_src, _src_len, _tgt = encode_batch(test_set[:2])
print("Batch of 2: src", tuple(_src.shape), "· tgt", tuple(_tgt.shape))
print("tgt[0] =", _tgt[0].tolist(), "->", repr(ids_to_string(_tgt[0, 1:].tolist())))

**The model (provided).** One class serves both models in this lab.

- **Encoder** (briefing eq. `encoder`): an embedding layer and a one-layer, unidirectional GRU. `encode` returns all the encoder states `h`, shape `(B, S, d_h)`, and the final state `h_S`, shape `(B, d_h)`. Padding is handled by packing, so `h_S` is the state after the last *real* character of each source, and `h` is zero at padded positions.
- **Decoder**: a second embedding layer and a second GRU, `model.dec`, with the same hidden size $d_h$, so that the dot-product score of Exercise 3 applies.
- **Output layer** `model.W_o`: a linear map from $d_h$ to the $|V| = 15$ logits (eq. `output`).
- With `attention="dot"` or `"additive"` the model also has the combination layer `model.W_c` of eq. `attentional`, and for `"additive"` the parameters `W_q`, `W_k`, `u` of eq. `additive`.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, attention=None, d_emb=D_EMB, d_h=D_H, d_a=64):
        super().__init__()
        assert attention in (None, "dot", "additive")
        self.attention = attention
        self.src_emb = nn.Embedding(len(SRC_ITOS), d_emb, padding_idx=PAD)  # e_{x_i}
        self.tgt_emb = nn.Embedding(len(TGT_ITOS), d_emb, padding_idx=PAD)  # e_{y_{t-1}}
        self.enc = nn.GRU(d_emb, d_h, batch_first=True)  # RNN_enc
        self.dec = nn.GRU(d_emb, d_h, batch_first=True)  # RNN_dec
        self.W_o = nn.Linear(d_h, len(TGT_ITOS))  # z_t = W_o s_t + b_o
        if attention is not None:
            self.W_c = nn.Linear(2 * d_h, d_h)  # s~_t = tanh(W_c [s_t; h_bar_t] + b_c)
        if attention == "additive":  # score = u^T tanh(W_q s_t + W_k h_i)
            self.W_q = nn.Parameter(torch.randn(d_a, d_h) / math.sqrt(d_h))
            self.W_k = nn.Parameter(torch.randn(d_a, d_h) / math.sqrt(d_h))
            self.u = nn.Parameter(torch.randn(d_a) / math.sqrt(d_a))

    def encode(self, src, src_len):
        """src: (B, S) ids. Returns h: (B, S, d_h), zero at padding, and h_S: (B, d_h)."""
        packed = nn.utils.rnn.pack_padded_sequence(
            self.src_emb(src), src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, h_last = self.enc(packed)  # h_last: (1, B, d_h), the state at each true end
        h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return h, h_last[0]


for _name in (None, "dot", "additive"):
    _n = sum(p.numel() for p in Seq2Seq(_name).parameters())
    print(f"attention = {str(_name):<8}  {_n:>7,} parameters")

# Part A · The encoder-decoder without attention

## Exercise 1 · Teacher-forced forward pass and loss (10 minutes)

Write the decoder of the plain model and its loss: briefing eqs. `decoder`, `output` and `loss`.

$$
s_0 = h_S, \qquad s_t = \mathrm{RNN}_{\mathrm{dec}}(s_{t-1}, e_{y_{t-1}}), \qquad z_t = W_o s_t + b_o, \qquad
\mathcal{L} = -\sum_{t} \log \mathrm{softmax}(z_t)_{y_t}.
$$

**Teacher forcing.** The batch tensor `tgt` holds `<bos> y_1 … y_T <eos>` and padding. The decoder input is the sequence shifted right, `tgt_in = tgt[:, :-1]` (it starts with `<bos>`), and the targets are `tgt_out = tgt[:, 1:]` (they end with `<eos>`). Position $t$ of `tgt_in` holds $y_{t-1}$ and position $t$ of `tgt_out` holds $y_t$. The training loop does the shifting; your functions receive the two halves.

**Predict.** Before any training, the logits are close to zero, so every target token gets probability about $1/|V|$. What mean loss per token do you expect, with $|V| = 15$?

**Task.**

- `forward_plain(model, src, src_len, tgt_in)`: encode the source, start the decoder from $s_0 = h_S$, run `model.dec` over the embedded `tgt_in` in one call, and return the logits, shape `(B, T, |V|)`. `nn.GRU` takes its initial state with shape `(1, B, d_h)`.
- `seq2seq_loss(logits, tgt_out)`: the mean cross-entropy over all target positions that are not padding. `F.cross_entropy` expects logits of shape `(N, |V|)` and targets of shape `(N,)`; its `ignore_index` argument skips padding.

In [ ]:
# TODO 1: complete both functions.
def forward_plain(model, src, src_len, tgt_in):
    """Teacher-forced logits of the model without attention. Returns (B, T, |V|)."""
    h, h_S = model.encode(src, src_len)  # h: (B, S, d_h) is not used without attention
    ...


def seq2seq_loss(logits, tgt_out):
    """Mean cross-entropy per target token, ignoring padding. logits: (B, T, |V|), tgt_out: (B, T)."""
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def forward_plain(model, src, src_len, tgt_in):
    """Teacher-forced logits of the model without attention. Returns (B, T, |V|)."""
    h, h_S = model.encode(src, src_len)  # h: (B, S, d_h) is not used without attention
    s, _ = model.dec(model.tgt_emb(tgt_in), h_S.unsqueeze(0))  # s_0 = h_S; s: (B, T, d_h)
    return model.W_o(s)  # z_t = W_o s_t + b_o: (B, T, |V|)


@workshop.solution(1)
def seq2seq_loss(logits, tgt_out):
    """Mean cross-entropy per target token, ignoring padding. logits: (B, T, |V|), tgt_out: (B, T)."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1),
                           ignore_index=PAD)

<details>
<summary>Why this solution works</summary>

Passing `h_S` as the GRU's initial state is eq. `decoder`'s $s_0 = h_S$: it is the **only** place where the source enters the plain model. Because the decoder input is the gold target shifted right (teacher forcing), all $T$ steps are known in advance and `nn.GRU` can run them in one call. `model.W_o` applied to every $s_t$ is eq. `output`.

`F.cross_entropy` applies the log-softmax and picks out $-\log P_\theta(y_t \mid y_{<t}, x)$ at the gold index, which is eq. `loss`. It averages over the non-padding positions instead of summing; the mean is the sum divided by the number of target tokens in the batch, so it has the same minimizer and its value is comparable across batches. Padding positions must be ignored, otherwise the model would be rewarded for predicting `<pad>` after `<eos>`.

An untrained model gives each of the 15 tokens probability close to $1/15$, so the loss starts near $\ln 15 \approx 2.71$ nats.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
torch.manual_seed(SEED)
_model = Seq2Seq(attention=None)
_src, _src_len, _tgt = encode_batch(random.Random(SEED).sample(train_set, 64))  # mixed K, so padded
_logits = forward_plain(_model, _src, _src_len, _tgt[:, :-1])
assert _logits.shape == (64, _tgt.size(1) - 1, len(TGT_ITOS)), _logits.shape
_loss = seq2seq_loss(_logits, _tgt[:, 1:])
print(f"Untrained loss {_loss.item():.4f} nats per token · ln|V| = {math.log(len(TGT_ITOS)):.4f}")
assert abs(_loss.item() - math.log(len(TGT_ITOS))) < 0.15, "an untrained model should be close to ln|V|"

# Padding is ignored: changing the logits at padded positions does not change the loss.
_pad = _tgt[:, 1:] == PAD
assert _pad.any()
_noisy = _logits.detach().clone()
_noisy[_pad] = 100 * torch.randn_like(_noisy[_pad])
assert torch.isclose(seq2seq_loss(_noisy, _tgt[:, 1:]), _loss.detach()), "padding must be ignored"

# The source enters only through h_S: two sources with the same final state give the same logits.
_h, _h_S = _model.encode(_src[:1], _src_len[:1])
_s, _ = _model.dec(_model.tgt_emb(_tgt[:1, :-1]), _h_S.unsqueeze(0))
assert torch.allclose(_logits[:1], _model.W_o(_s), atol=1e-6), "start the decoder from h_S"
print("Checkpoint 1 passed")

**Run: train the model without attention (provided).** The training loop is the same for every model in this lab:

- the same `STEPS` steps of `BATCH` sequence pairs, drawn from the training set in the **same order** for every model (the batch order has its own seed);
- Adam with learning rate `LR`, and gradient clipping at `CLIP` (Lab 3);
- the loss is your `seq2seq_loss` on the logits from the model's forward function.

So the comparison in Exercise 3 is under an **equal budget**. Start the training cell now; it prints the loss as it goes. Read Exercise 2 while it runs.

The next cell also defines the decoding helpers. `decoder_step` advances the decoder by one token; `greedy_decode` feeds the most probable token back in at each step until `<eos>` (briefing, Section 7). `readout` turns decoder states into logits: $z_t = W_o s_t$ without attention, and eq. `attentional` with it (that branch is used from Exercise 3 on).

In [ ]:
MAX_OUT = 4 * 11 + 1  # the longest target (4 ISO dates and 3 spaces) plus <eos>


def readout(model, s, h, mask):
    """Logits from decoder states s: (B, T, d_h). Returns (logits, alpha or None)."""
    if model.attention is None:
        return model.W_o(s), None  # eq. output
    if model.attention == "dot":
        alpha, h_bar = dot_product_attention(s, h, mask)  # Exercise 3
    else:
        alpha, h_bar = masked_attention(additive_score(s, h, model.W_q, model.W_k, model.u), h, mask)
    s_tilde = torch.tanh(model.W_c(torch.cat([s, h_bar], dim=-1)))  # eq. attentional
    return model.W_o(s_tilde), alpha  # alpha: (B, T, S)


def train(model, forward_fn, steps=STEPS, log_every=None):
    """Train with Adam and clipping on seeded batches. Returns (losses, seconds)."""
    model.to(device).train()
    batches = random.Random(SEED + 2)  # the same batch order for every model
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    log_every = log_every or max(steps // 8, 1)
    losses, started = [], time.time()
    for step in range(1, steps + 1):
        src, src_len, tgt = encode_batch(batches.sample(train_set, BATCH))
        src, tgt = src.to(device), tgt.to(device)
        logits = forward_fn(model, src, src_len, tgt[:, :-1])
        if isinstance(logits, tuple):  # the attention forward also returns alpha
            logits = logits[0]
        loss = seq2seq_loss(logits, tgt[:, 1:])
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        losses.append(loss.item())
        if step % log_every == 0 or step == steps:
            recent = np.mean(losses[-log_every:])
            print(f"  step {step:>5}/{steps}  loss {recent:.4f}  ({time.time() - started:.0f} s)")
    model.eval()
    return losses, time.time() - started


def decoder_step(model, y_prev, state, h, mask):
    """One decoder step for n sequences. y_prev: (n,) ids; state: (1, n, d_h);
    h: (n, S, d_h); mask: (n, S). Returns (log-probs (n, |V|), new state, alpha (n, S) or None)."""
    s, state = model.dec(model.tgt_emb(y_prev).unsqueeze(1), state)  # s: (n, 1, d_h)
    logits, alpha = readout(model, s, h, mask)
    return (F.log_softmax(logits[:, 0], dim=-1), state,
            None if alpha is None else alpha[:, 0])


@torch.no_grad()
def greedy_decode(model, examples, batch_size=500):
    """Greedy decoding. Returns (predicted strings, attention matrices or None).

    Each attention matrix is a NumPy array (steps, S): one row per generated token,
    including <eos>, one column per source character."""
    model.eval()
    preds, alphas = [], []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start : start + batch_size]
        src, src_len, _ = encode_batch(chunk)
        src = src.to(device)
        h, h_S = model.encode(src, src_len)
        mask = src != PAD
        y, state = torch.full((len(chunk),), BOS, device=device), h_S.unsqueeze(0)
        out, att = [], []
        for _ in range(MAX_OUT):
            log_p, state, alpha = decoder_step(model, y, state, h, mask)
            y = log_p.argmax(dim=-1)
            out.append(y)
            att.append(alpha)
        out = torch.stack(out, dim=1).tolist()  # (B, MAX_OUT)
        for b, ex in enumerate(chunk):
            preds.append(ids_to_string(out[b]))
            if model.attention is not None:
                steps = min(len(preds[-1]) + 1, MAX_OUT)  # generated tokens, with <eos>
                rows = torch.stack([a[b, : len(ex["src"])] for a in att[:steps]])
                alphas.append(rows.cpu().numpy())
    return preds, (alphas if model.attention is not None else None)

In [ ]:
torch.manual_seed(SEED)
model_plain = Seq2Seq(attention=None)
print(f"Training the model WITHOUT attention for {STEPS} steps on {device}")
losses_plain, seconds_plain = train(model_plain, forward_plain)
print(f"Done in {seconds_plain:.0f} s")

## Exercise 2 · Exact match by input length (8 minutes)

A date answer is either right or wrong: `2021-03-04` for `2021-03-03` is wrong, however close the characters are. The metric of this lab is **exact match**: the share of examples whose whole output string equals the target.

**Predict.** Write down your guess for the exact-match accuracy of the model without attention in each bucket $K = 1, 2, 3, 4$. Which bucket will fail? Will the model get the *first* or the *last* date of a long input right more often?

**Task.** Complete `exact_match(preds, golds)`: two lists of strings of the same length; return the share of positions where they are equal, as a float.

In [ ]:
# TODO 2: complete this function.
def exact_match(preds, golds):
    """Share of predictions equal to their gold string. Two lists of strings -> float."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def exact_match(preds, golds):
    """Share of predictions equal to their gold string. Two lists of strings -> float."""
    assert len(preds) == len(golds)
    return sum(p == g for p, g in zip(preds, golds)) / len(golds)

<details>
<summary>Why this solution works</summary>

String equality is all-or-nothing, which is the right standard for a date: one wrong digit makes the answer wrong. The metric is strict on purpose. A source with $K$ dates has $10K + K - 1$ target characters, and every one must be right, so a model with a fixed per-character error rate would score lower on long inputs anyway. The per-date breakdown below separates the two effects.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
assert exact_match(["2021-03-03"], ["2021-03-03"]) == 1.0
assert exact_match(["2021-03-04", "1999-12-31"], ["2021-03-03", "1999-12-31"]) == 0.5
assert exact_match(["2021-03-03 1999-12-31"], ["2021-03-03"]) == 0.0, "extra output is wrong"
assert exact_match(["", "x", "y"], ["a", "x", "y"]) == 2 / 3
assert isinstance(exact_match(["a"], ["a"]), float)
print("Checkpoint 2 passed")

In [ ]:
def accuracy_by_bucket(preds, examples):
    """{K: exact match on the test sources with K dates}."""
    return {K: exact_match([p for p, ex in zip(preds, examples) if ex["K"] == K],
                           [ex["tgt"] for ex in examples if ex["K"] == K])
            for K in (1, 2, 3, 4)}


def per_date_accuracy(preds, examples, K):
    """For sources with K dates: share of examples whose k-th output date is right, k = 1..K."""
    pairs = [(p.split(" "), ex["tgt"].split(" ")) for p, ex in zip(preds, examples) if ex["K"] == K]
    return [float(np.mean([len(p) > k and p[k] == g[k] for p, g in pairs])) for k in range(K)]


SOURCE_CHARS = {K: np.mean([len(ex["src"]) for ex in test_set if ex["K"] == K]) for K in (1, 2, 3, 4)}
started = time.time()
preds_plain, _ = greedy_decode(model_plain, test_set)
acc_plain = accuracy_by_bucket(preds_plain, test_set)
print(f"Model without attention, test set, greedy decoding ({time.time() - started:.1f} s)\n")
print(" K  source characters  exact match")
for K in (1, 2, 3, 4):
    print(f"{K:>2}  {SOURCE_CHARS[K]:>17.1f}  {acc_plain[K]:>11.3f}")

print("\nShare of examples with the k-th output date right, k = 1..K")
for K in (2, 3, 4):
    print(f"  K = {K}:  " + "  ".join(f"{a:.3f}" for a in per_date_accuracy(preds_plain, test_set, K)))
for K in (2, 4):
    print(f"\nTwo errors with K = {K}:")
    for p, ex in [(p, ex) for p, ex in zip(preds_plain, test_set) if ex["K"] == K and p != ex["tgt"]][:2]:
        print(f"  source {ex['src']!r}\n  gold   {ex['tgt']}\n  output {p}")

**Explain.** Compare the table with your prediction.

- On one date the plain model is right about nine times in ten: the task itself is learnable. With two or more dates it is essentially never right. The model has the same $d_h = 128$ numbers in $h_S$ whether the source holds one date or four (the fixed-vector bottleneck, briefing Section 3).
- Look at the per-date lines and the errors. The outputs have the right *shape*: the right number of dates, each a well-formed ISO date. The first date is right about as often as a single date is; every later date is wrong. At this budget, what the model has learned to carry through $h_S$ is one date. The rest of the output is a plausible guess.

**What this does and does not show.** It shows that *at this budget* (this $d_h$, these steps, this optimizer) the plain model fails on long inputs. A larger $d_h$ or much longer training would raise the numbers, because four dates do fit in 128 numbers in principle. The claim of this lab is about equal budgets: the next model gets exactly the same ones.

# Part B · Attention

## Exercise 3 · Dot-product attention (12 minutes)

Keep all encoder states $h_1, \dots, h_S$, and at each decoder step $t$ read a weighted average of them (briefing, Section 4):

$$
\mathrm{score}(s_t, h_i) = s_t^\top h_i, \qquad
\alpha_{t,i} = \frac{\exp \mathrm{score}(s_t, h_i)}{\sum_{j=1}^{S} \exp \mathrm{score}(s_t, h_j)}, \qquad
\bar h_t = \sum_{i=1}^{S} \alpha_{t,i}\, h_i .
$$

These are eqs. `dot`, `weights` and `context`. Padded source positions must get weight exactly zero: set their scores to $-\infty$ before the softmax.

The code works on all decoder steps at once: the queries `q` have shape `(B, T, d_h)`, one row per decoder step (during greedy decoding $T = 1$).

**Predict.** For one query, the scores over three source positions are $(2, 0, 0)$. What are the three weights? What happens to them if a fourth, padded position has score $5$ before masking?

**Task.** Complete two functions. The second one should be one line that calls the first.

- `masked_attention(scores, keys, mask)`: from scores `(B, T, S)`, keys `(B, S, d_h)` and a boolean mask `(B, S)` (`True` for real tokens), return `alpha`, shape `(B, T, S)`, and the context `h_bar`, shape `(B, T, d_h)`. Useful: `masked_fill`, `torch.softmax(..., dim=-1)`, the batched matrix product `@`.
- `dot_product_attention(q, keys, mask)`: the dot-product scores $s_t^\top h_i$ for every pair $(t, i)$, passed to `masked_attention`.

Exercise 4 reuses `masked_attention` with a different score.

In [ ]:
# TODO 3: complete both functions.
def masked_attention(scores, keys, mask):
    """scores: (B, T, S), keys: (B, S, d_h), mask: (B, S) bool, True = real token.
    Returns alpha: (B, T, S) and h_bar: (B, T, d_h)."""
    raise NotImplementedError("TODO 3")


def dot_product_attention(q, keys, mask):
    """Dot-product attention. q: (B, T, d_h) queries s_t; keys: (B, S, d_h) encoder states h_i."""
    raise NotImplementedError("TODO 3")

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def masked_attention(scores, keys, mask):
    """scores: (B, T, S), keys: (B, S, d_h), mask: (B, S) bool, True = real token.
    Returns alpha: (B, T, S) and h_bar: (B, T, d_h)."""
    scores = scores.masked_fill(~mask.unsqueeze(1), float("-inf"))  # padded positions: exp(-inf) = 0
    alpha = torch.softmax(scores, dim=-1)  # softmax over the S source positions (eq. weights)
    h_bar = alpha @ keys  # (B, T, S) @ (B, S, d_h) -> (B, T, d_h), sum_i alpha_ti h_i (eq. context)
    return alpha, h_bar


@workshop.solution(3)
def dot_product_attention(q, keys, mask):
    """Dot-product attention. q: (B, T, d_h) queries s_t; keys: (B, S, d_h) encoder states h_i."""
    return masked_attention(q @ keys.transpose(1, 2), keys, mask)  # scores s_t^T h_i: (B, T, S)

<details>
<summary>Why this solution works</summary>

`q @ keys.transpose(1, 2)` computes every dot product $s_t^\top h_i$ in one batched matrix product: entry $(b, t, i)$ is the score of decoder step $t$ against source position $i$ (eq. `dot`). The softmax runs over the **last** axis, the $S$ source positions, not over the vocabulary (eq. `weights`). The mask has shape `(B, S)`; `unsqueeze(1)` makes it `(B, 1, S)`, so the same source mask applies to every decoder step. Since $\exp(-\infty) = 0$, a padded position gets weight exactly 0 and the remaining weights still sum to 1.

`alpha @ keys` is the weighted average $\sum_i \alpha_{t,i} h_i$ (eq. `context`) for every step at once. For the worked example, the weights are $e^2 / (e^2 + 2) \approx 0.787$ and $1 / (e^2 + 2) \approx 0.106$ twice. A padded fourth position with score 5 would take most of the weight if it were not masked; masked, it changes nothing.

</details>

In [ ]:
# Checkpoint 3
# The briefing's worked example: scores (2, 0, 0) over three keys, plus one padded key.
workshop.checkpoint(3)
_keys = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [0.0, -1.0], [9.0, 9.0]]])  # (1, S=4, 2)
_mask = torch.tensor([[True, True, True, False]])
_q = torch.tensor([[[2.0, 0.0]]])  # (1, T=1, 2): scores 2, 0, 0 and 18 (padded)
_alpha, _h_bar = dot_product_attention(_q, _keys, _mask)
assert _alpha.shape == (1, 1, 4) and _h_bar.shape == (1, 1, 2), (_alpha.shape, _h_bar.shape)
_e2 = math.exp(2)
assert torch.allclose(_alpha[0, 0], torch.tensor([_e2, 1, 1, 0]) / (_e2 + 2), atol=1e-6), _alpha
assert _alpha[0, 0, 3].item() == 0.0, "a padded position must get weight exactly 0"
assert torch.allclose(_h_bar[0, 0], torch.tensor([_e2 / (_e2 + 2), 0.0]), atol=1e-6), _h_bar

# Random batch with padding, against PyTorch's attention with the scaling turned off.
_g = torch.Generator().manual_seed(SEED)
_q, _keys = torch.randn(3, 5, 8, generator=_g), torch.randn(3, 7, 8, generator=_g)
_mask = torch.arange(7)[None, :] < torch.tensor([[7], [4], [1]])  # true lengths 7, 4, 1
_alpha, _h_bar = dot_product_attention(_q, _keys, _mask)
assert _alpha.shape == (3, 5, 7) and _h_bar.shape == (3, 5, 8)
assert torch.allclose(_alpha.sum(-1), torch.ones(3, 5), atol=1e-6), "each row must sum to 1"
assert (_alpha[~_mask.unsqueeze(1).expand(-1, 5, -1)] == 0).all(), "padded weights must be 0"
_ref = F.scaled_dot_product_attention(_q, _keys, _keys, attn_mask=_mask.unsqueeze(1), scale=1.0)
assert torch.allclose(_h_bar, _ref, atol=1e-5), "context differs from the reference"
print("Checkpoint 3 passed")

**Run: train the model with dot-product attention (provided).** The forward pass is the one you wrote in Exercise 1 plus the attention step: the same encoder, the same decoder started from $s_0 = h_S$, then your `dot_product_attention` with the decoder states as queries, and the combination layer of eq. `attentional` before the output layer (`readout`, defined above). Same $d_h$, same steps, same optimizer, same batches.

Start the cell, then go on to Exercise 4 while it trains. Come back for the comparison.

In [ ]:
def forward_attention(model, src, src_len, tgt_in):
    """Teacher-forced logits (B, T, |V|) and attention weights (B, T, S) of an attention model."""
    h, h_S = model.encode(src, src_len)  # h: (B, S, d_h)
    s, _ = model.dec(model.tgt_emb(tgt_in), h_S.unsqueeze(0))  # as in forward_plain
    return readout(model, s, h, src != PAD)  # attention, then eq. attentional, then W_o


torch.manual_seed(SEED)
model_dot = Seq2Seq(attention="dot")
print(f"Training the model WITH dot-product attention for {STEPS} steps on {device}")
losses_dot, seconds_dot = train(model_dot, forward_attention)
print(f"Done in {seconds_dot:.0f} s")

In [ ]:
started = time.time()
preds_dot, alphas_dot = greedy_decode(model_dot, test_set)
acc_dot = accuracy_by_bucket(preds_dot, test_set)
print(f"Test set, exact match, greedy decoding ({time.time() - started:.1f} s)\n")
print(" K  source characters  no attention  dot-product attention")
for K in (1, 2, 3, 4):
    print(f"{K:>2}  {SOURCE_CHARS[K]:>17.1f}  {acc_plain[K]:>12.3f}  {acc_dot[K]:>21.3f}")
print("\nK = 4 sources, share with the k-th output date right")
for name, preds in [("no attention", preds_plain), ("attention", preds_dot)]:
    print(f"  {name:<13}" + "  ".join(f"{a:.3f}" for a in per_date_accuracy(preds, test_set, 4)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ks = [1, 2, 3, 4]
ax1.plot(ks, [acc_plain[K] for K in ks], "o-", color=NAVY, linewidth=2, label="no attention")
ax1.plot(ks, [acc_dot[K] for K in ks], "s-", color=ACCENT, linewidth=2, label="dot-product attention")
ax1.set_xticks(ks, [f"K = {K}\n{SOURCE_CHARS[K]:.0f} chars" for K in ks])
ax1.set_ylim(0, 1.02)
ax1.set_ylabel("exact match (test)")
ax1.set_title("Accuracy by input length")
ax1.legend(frameon=False)
window = max(STEPS // 50, 1)
for losses, color, name in [(losses_plain, NAVY, "no attention"), (losses_dot, ACCENT, "attention")]:
    smooth = np.convolve(losses, np.ones(window) / window, mode="valid")
    ax2.plot(np.arange(len(smooth)) + window, smooth, color=color, linewidth=2, label=name)
ax2.set_yscale("log")
ax2.set_xlabel("training step")
ax2.set_ylabel("training loss (nats per token)")
ax2.set_title("Same budget: same steps, batches, optimizer")
ax2.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Checkpoint: the bottleneck, measured. The thresholds are the lab's targets; they hold
# with margin at the configured budget (see the text below).
workshop.checkpoint(label="bottleneck")
if QUICK:
    print(f"QUICK = True ({STEPS} of {STEPS_FULL} steps): accuracy targets not checked.")
else:
    # The lab's target for K = 1 is 0.90. Measured at 4,000 steps on CPU: 0.898 on test in one
    # run, 0.99 on a validation set in another run with the same settings (floating-point
    # nondeterminism makes runs diverge). The assertion leaves a margin below the target.
    assert acc_plain[1] >= 0.85, f"the plain model should handle one date: {acc_plain[1]:.3f}"
    assert acc_plain[4] <= 0.50, f"the plain model should fail on four dates: {acc_plain[4]:.3f}"
    assert acc_dot[4] >= 0.90, f"attention should handle four dates: {acc_dot[4]:.3f}"
    print("Checkpoint passed: same budget, the plain model fails on long inputs and attention does not.")

**Explain.**

- The two models differ only in how the decoder reads the source. Without attention, everything passes through $h_S$; with attention, every step reads a fresh weighted average of all $S$ encoder states, so the memory grows with the source (briefing, Section 4, "Why this fixes the bottleneck").
- The training losses tell the same story: with the same batches and steps, the attention model's loss falls far below the plain model's within the first few hundred steps.
- The plain model's numbers are not its ceiling. With more capacity or training it would do better on $K = 4$; the claim is that, **at an equal budget**, attention removes the failure. The attention model costs $S \times T$ extra scores per sequence pair and one extra layer ($W_c$, about 33,000 parameters).

## Exercise 4 · The additive score (10 minutes)

Bahdanau et al. (2015) score each pair with a small feed-forward network (briefing eq. `additive`):

$$
\mathrm{score}(s_t, h_i) = u^\top \tanh\big(W_q s_t + W_k h_i\big),
$$

with $W_q, W_k \in \mathbb{R}^{d_a \times d_h}$ and $u \in \mathbb{R}^{d_a}$. Everything after the score (the masked softmax and the weighted average) is your `masked_attention` from Exercise 3.

**Predict.** The dot-product score needs one matrix product for all pairs $(t, i)$. How large is the intermediate tensor of the additive score for a batch of shape $B \times T \times S$, with attention size $d_a$?

**Task.** Complete `additive_score(q, keys, W_q, W_k, u)`, returning scores of shape `(B, T, S)`. Project the queries and the keys **once** each (`q @ W_q.T` has shape `(B, T, d_a)`), then add them with broadcasting: insert a size-1 axis so the shapes become `(B, T, 1, d_a)` and `(B, 1, S, d_a)`.

In [ ]:
# TODO 4: complete this function.
def additive_score(q, keys, W_q, W_k, u):
    """u^T tanh(W_q s_t + W_k h_i) for every pair (t, i).
    q: (B, T, d_h), keys: (B, S, d_h), W_q and W_k: (d_a, d_h), u: (d_a,). Returns (B, T, S)."""
    raise NotImplementedError("TODO 4")

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def additive_score(q, keys, W_q, W_k, u):
    """u^T tanh(W_q s_t + W_k h_i) for every pair (t, i).
    q: (B, T, d_h), keys: (B, S, d_h), W_q and W_k: (d_a, d_h), u: (d_a,). Returns (B, T, S)."""
    q_proj = (q @ W_q.T).unsqueeze(2)  # W_q s_t: (B, T, 1, d_a)
    k_proj = (keys @ W_k.T).unsqueeze(1)  # W_k h_i: (B, 1, S, d_a)
    return torch.tanh(q_proj + k_proj) @ u  # (B, T, S, d_a) @ (d_a,) -> (B, T, S)

<details>
<summary>Why this solution works</summary>

Projecting once costs $O((T + S)\,d_a\,d_h)$ instead of recomputing $W_k h_i$ for every decoder step. Broadcasting `(B, T, 1, d_a) + (B, 1, S, d_a)` then forms all $T \times S$ sums $W_q s_t + W_k h_i$, which is the "additive" part. The `tanh` is elementwise, and `@ u` contracts the last axis, reducing each $d_a$-vector to one number. The intermediate tensor has $B \times T \times S \times d_a$ entries: for a batch of 128 sources with four dates, $T \approx 44$, $S \approx 74$ and $d_a = 64$, that is about 27 million numbers, where the dot product needed only the $B \times T \times S$ scores. This is the cost the briefing's table refers to, and the reason Module 5 keeps the multiplicative form.

</details>

In [ ]:
# Checkpoint 4
workshop.checkpoint(4)
_g = torch.Generator().manual_seed(SEED)
_B, _T, _S, _d_h, _d_a = 2, 3, 5, 4, 6
_q, _keys = torch.randn(_B, _T, _d_h, generator=_g), torch.randn(_B, _S, _d_h, generator=_g)
_W_q, _W_k = torch.randn(_d_a, _d_h, generator=_g), torch.randn(_d_a, _d_h, generator=_g)
_u = torch.randn(_d_a, generator=_g)
_scores = additive_score(_q, _keys, _W_q, _W_k, _u)
assert _scores.shape == (_B, _T, _S), _scores.shape

# Reference: the equation, one pair at a time.
_ref = torch.empty(_B, _T, _S)
for b in range(_B):
    for t in range(_T):
        for i in range(_S):
            _ref[b, t, i] = _u @ torch.tanh(_W_q @ _q[b, t] + _W_k @ _keys[b, i])
assert torch.allclose(_scores, _ref, atol=1e-5), "scores differ from u^T tanh(W_q s_t + W_k h_i)"

# Plugged into the attention step of Exercise 3.
_mask = torch.tensor([[True] * 5, [True] * 3 + [False] * 2])
_alpha, _h_bar = masked_attention(_scores, _keys, _mask)
assert torch.allclose(_alpha.sum(-1), torch.ones(_B, _T), atol=1e-6), "weights must sum to 1"
assert (_alpha[1, :, 3:] == 0).all() and _h_bar.shape == (_B, _T, _d_h)
print("Checkpoint 4 passed")

# Optional: train the additive model with the same budget. Off by default to keep the lab
# within its time budget; it costs about as much as one more training run.
TRAIN_ADDITIVE = False  # set to True to train it
if TRAIN_ADDITIVE:
    torch.manual_seed(SEED)
    model_add = Seq2Seq(attention="additive")
    losses_add, seconds_add = train(model_add, forward_attention)
    acc_add = accuracy_by_bucket(greedy_decode(model_add, test_set)[0], test_set)
    print("Additive attention, exact match by K:", {K: round(a, 3) for K, a in acc_add.items()})

# Part C · Reading attention

## Exercise 5 · Attention heat-maps and the alignment hit rate (7 minutes)

Collect the weights $\alpha_{t,i}$ into a matrix: one row per output token $t$, one column per source character $i$ (briefing, Section 6). Each row sums to 1. The next cell plots this matrix for the briefing's example and three test sources, as decoded by the attention model.

**Predict.** For `3 March 2021` → `2021-03-03`: where should the bright cells be while the decoder writes `2021`? While it writes `03` for the month? Will the bright cells sit *on* the characters you expect, or one position later?

In [ ]:
def plot_attention(alpha, source, output, ax, title=""):
    """Heat-map of alpha (rows: output tokens incl. <eos>; columns: source characters)."""
    ax.imshow(alpha, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(source)), [ch if ch != " " else "␣" for ch in source], fontsize=8)
    ax.set_yticks(range(len(alpha)), (list(output) + ["<eos>"])[: len(alpha)], fontsize=8)
    ax.set_xlabel("source position i")
    ax.set_ylabel("output step t")
    ax.set_title(title, fontsize=10)


showcase_pred, showcase_alpha = greedy_decode(model_dot, [SHOWCASE])
showcase_pred, showcase_alpha = showcase_pred[0], showcase_alpha[0]
print(f"{SHOWCASE['src']!r} -> {showcase_pred!r} ({'correct' if showcase_pred == SHOWCASE['tgt'] else 'WRONG'})")

picks = [next(n for n, ex in enumerate(test_set) if ex["K"] == K) for K in (1, 2, 3)]
fig = plt.figure(figsize=(13, 9))
grid = fig.add_gridspec(2, 3, height_ratios=[1, 1.4])
plot_attention(showcase_alpha, SHOWCASE["src"], showcase_pred, fig.add_subplot(grid[0, 0]),
               "the briefing's example")
for col, n in enumerate(picks[:2], start=1):
    plot_attention(alphas_dot[n], test_set[n]["src"], preds_dot[n], fig.add_subplot(grid[0, col]),
                   f"K = {test_set[n]['K']}")
n = picks[2]
plot_attention(alphas_dot[n], test_set[n]["src"], preds_dot[n], fig.add_subplot(grid[1, :]),
               f"K = {test_set[n]['K']}")
plt.tight_layout()
plt.show()

**Task.** A heat-map is one example. Put a number on all of them. Complete `alignment_hit_rate(alphas, examples, tol)`: over every **digit** of every target, the share for which the source position with the largest weight lies inside the span of the date that digit belongs to.

- `alphas[n]` is an array of shape `(T, S)` for example `n`. Row `t` holds the weights used while predicting target character `t` (these come from a teacher-forced pass over the gold target, so row `t` always lines up with character `t`).
- `examples[n]["tgt"]` is the target string. Character `t` belongs to date `t // 11`: each ISO date has 10 characters, followed by a space.
- `examples[n]["spans"][k] = (start, end)` is the span of date `k` in the source, end exclusive.
- An arg-max at most `tol` positions **after** the end of the span also counts. The encoder is unidirectional, so $h_i$ summarizes the source up to position $i$ (briefing, Section 6): the state just after a date can carry it.

In [ ]:
# TODO 5: complete this function.
def alignment_hit_rate(alphas, examples, tol=1):
    """Share of target digits whose arg-max source position is inside their date's span
    (or at most `tol` positions after it)."""
    hits = total = 0
    for alpha, ex in zip(alphas, examples):
        for t, ch in enumerate(ex["tgt"]):
            ...
    return hits / total

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def alignment_hit_rate(alphas, examples, tol=1):
    """Share of target digits whose arg-max source position is inside their date's span
    (or at most `tol` positions after it)."""
    hits = total = 0
    for alpha, ex in zip(alphas, examples):
        for t, ch in enumerate(ex["tgt"]):
            if not ch.isdigit():  # skip the '-' and the spaces between dates
                continue
            start, end = ex["spans"][t // 11]  # the date this digit belongs to
            i = int(np.argmax(alpha[t]))  # where the decoder read most while writing it
            hits += start <= i < end + tol
            total += 1
    return hits / total

<details>
<summary>Why this solution works</summary>

Each digit of the target is copied or looked up from exactly one date of the source, so "the arg-max lies in that date's span" is a checkable version of "the soft alignment points at the right place". Only digits are scored: the `-` and the spaces are fixed by the format and need no source position. The tolerance allows for the unidirectional encoder, whose state $h_i$ summarizes positions $1, \dots, i$.

A hit rate says nothing about *how* the context is used, only where the decoder read (briefing, Section 6, "The weights show where the model read, not why").

</details>

In [ ]:
# Checkpoint 5
workshop.checkpoint(5)
_ex = {"tgt": "2021-03-03 1999-12-31", "spans": [(0, 12), (17, 30)]}
_alpha = np.zeros((len(_ex["tgt"]), 32))
_alpha[:, 5] = 1.0  # always reads position 5, inside date 0 only
assert math.isclose(alignment_hit_rate([_alpha], [_ex]), 0.5), "8 of 16 digits are in date 0"
_alpha[11:, :] = 0
_alpha[11:, 30] = 1.0  # date 1 read one position after its span
assert alignment_hit_rate([_alpha], [_ex], tol=1) == 1.0
assert alignment_hit_rate([_alpha], [_ex], tol=0) == 0.5

@torch.no_grad()
def teacher_forced_attention(model, examples, batch_size=500):
    """alpha for each example on its gold target: arrays (T + 1, S), rows for y_1..y_T and <eos>."""
    out = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start : start + batch_size]
        src, src_len, tgt = encode_batch(chunk)
        _, alpha = forward_attention(model, src.to(device), src_len, tgt[:, :-1].to(device))
        for b, ex in enumerate(chunk):
            out.append(alpha[b, : len(ex["tgt"]) + 1, : len(ex["src"])].cpu().numpy())
    return out


tf_alphas = teacher_forced_attention(model_dot, test_set)
print(" K  hit rate (tol = 0)  hit rate (tol = 1)  chance (tol = 1)")
hit_rates = {}
for K in (1, 2, 3, 4):
    idx = [n for n, ex in enumerate(test_set) if ex["K"] == K]
    sub_a, sub_e = [tf_alphas[n] for n in idx], [test_set[n] for n in idx]
    hit_rates[K] = alignment_hit_rate(sub_a, sub_e, tol=1)
    # chance: an arg-max drawn uniformly over the source, averaged the same way
    chance = np.mean([(min(ex["spans"][t // 11][1] + 1, len(ex["src"])) - ex["spans"][t // 11][0])
                      / len(ex["src"]) for ex in sub_e for t, ch in enumerate(ex["tgt"]) if ch.isdigit()])
    print(f"{K:>2}  {alignment_hit_rate(sub_a, sub_e, tol=0):>18.3f}  {hit_rates[K]:>18.3f}  {chance:>16.3f}")
HIT_RATE = alignment_hit_rate(tf_alphas, test_set, tol=1)
print(f"\nAll test sources, tol = 1: {HIT_RATE:.3f}")
if QUICK:
    print("QUICK = True: hit-rate target not checked. Checkpoint 5 (function) passed")
else:
    assert HIT_RATE >= 0.95, f"the arg-max should usually fall in the right date: {HIT_RATE:.3f}"
    print("Checkpoint 5 passed")

**Explain.**

- Compare the hit rate with chance. The decoder reads from the right date almost every time, although no alignment was ever given to it: the weights were learned from the cross-entropy loss alone (briefing, Section 6).
- In the heat-maps, the rows for the year sit on the year of the source and the rows for the month sit on the month name, so the bright cells leave the diagonal when the source order differs from the ISO order. The weights are close to one-hot: this model reads one encoder state at a time.
- Look closely at the briefing's example. While the decoder writes a digit of the year, the brightest cell sits on the matching source digit or one position *after* it; for the month it sits inside or just after `March`, at the first state that has seen enough of the word; and in the recorded run (the briefing's figure) the day's leading `0` was read at the space after `3`, the first state that shows the day has one digit. This is the unidirectional encoder at work: $h_i$ summarizes the source up to position $i$, so the state that "knows" a character can be the one after it (briefing, Section 6, "$h_i$ is not the token $x_i$"). Compare the `tol = 0` and `tol = 1` columns for how much this matters at the level of whole dates.

## The results card

Every number below was measured in this run. The cell writes two files to the working directory:

- `lab04_results.json`: the settings, run times and exact match by bucket for both models;
- `lab04_attention_example.json`: the measured attention matrix for `3 March 2021`, with its source and output tokens. The briefing's attention-alignment figure is drawn from this file.

In [ ]:
settings = {"d_emb": D_EMB, "d_h": D_H, "batch": BATCH, "lr": LR, "clip": CLIP, "steps": STEPS,
            "optimizer": "Adam", "encoder": "1-layer unidirectional GRU", "decoder": "1-layer GRU, s_0 = h_S",
            "train_per_K": 25_000, "test_per_K": 500, "data_seeds": {"test": 0, "train": 1, "batches": SEED + 2},
            "quick": QUICK, "seed": SEED, "device": str(device), "torch": torch.__version__}
card = {
    "settings": settings,
    "source_chars_mean": {K: round(float(SOURCE_CHARS[K]), 1) for K in (1, 2, 3, 4)},
    "exact_match": {"no_attention": {K: round(acc_plain[K], 4) for K in (1, 2, 3, 4)},
                    "dot_attention": {K: round(acc_dot[K], 4) for K in (1, 2, 3, 4)}},
    "alignment_hit_rate_tol1": {"all": round(HIT_RATE, 4), **{K: round(hit_rates[K], 4) for K in (1, 2, 3, 4)}},
    "train_seconds": {"no_attention": round(seconds_plain, 1), "dot_attention": round(seconds_dot, 1)},
}
with open("lab04_results.json", "w") as f:
    json.dump(card, f, indent=2)

example = {
    "source": SHOWCASE["src"], "gold": SHOWCASE["tgt"], "prediction": showcase_pred,
    "correct": showcase_pred == SHOWCASE["tgt"],
    "source_tokens": list(SHOWCASE["src"]),
    "output_tokens": list(showcase_pred) + ["<eos>"],
    "alpha": [[round(float(a), 4) for a in row] for row in showcase_alpha],
    "layout": "alpha[t][i]: output step t (rows, with <eos>), source character i (columns); greedy decoding; rows sum to 1",
    "model": "dot-product attention, settings as in lab04_results.json",
    "settings": settings,
}
with open("lab04_attention_example.json", "w") as f:
    json.dump(example, f, indent=1)

print(json.dumps(card["exact_match"], indent=1))
print(f"Training: {seconds_plain:.0f} s without attention, {seconds_dot:.0f} s with · "
      f"whole lab so far {time.time() - LAB_START:.0f} s")
print("Wrote lab04_results.json and lab04_attention_example.json. This is the end of the core path.")

## Stretch (optional) · Beam search

For those who finish early. Nothing later in the workshop depends on this section.

Greedy decoding commits to the most probable token at every step. **Beam search** keeps the $B$ best partial outputs (**hypotheses**) instead (briefing, Section 7, and the optional box there). A hypothesis's **value** is its log-probability so far, $\sum_{\tau} \log P_\theta(y_\tau \mid y_{<\tau}, x)$. At each step:

1. extend every live hypothesis by every token in $V$, adding that token's log-probability to the value;
2. keep the $B$ candidates with the highest values;
3. a candidate that ends in `<eos>` is finished: set it aside, and continue with the rest;
4. stop when $B$ hypotheses have finished, when none is left alive, or at `max_len`.

Return the finished hypothesis with the best **length-normalized** value, value divided by the number of tokens (including `<eos>`). If none finished, use the live ones.

**Predict.** Will $B = 5$ be more accurate than greedy decoding on this task? Will its outputs have a higher log-probability?

**Task.** Complete `beam_search(model, source, B)` for one source string. It returns the output string and its (unnormalized) log-probability. Use the provided `decoder_step`: it runs the decoder for $n$ hypotheses at once, so stack the live hypotheses into one batch.

In [ ]:
# TODO 6 (stretch): complete this function.
@torch.no_grad()
def beam_search(model, source, B, max_len=MAX_OUT):
    """Beam search with length normalization for one source string. Returns (output, log-prob)."""
    model.eval()
    src, src_len, _ = encode_batch([{"src": source, "tgt": ""}])
    src = src.to(device)
    h, h_S = model.encode(src, src_len)  # h: (1, S, d_h)
    mask = src != PAD  # (1, S)
    alive = [([], 0.0)]  # hypotheses: (token ids so far, log-probability)
    state = h_S.unsqueeze(0)  # (1, n_alive, d_h): one decoder state per live hypothesis
    finished = []
    for _ in range(max_len):
        n = len(alive)
        y_prev = torch.tensor([ids[-1] if ids else BOS for ids, _ in alive], device=device)
        log_p, state, _ = decoder_step(model, y_prev, state, h.expand(n, -1, -1), mask.expand(n, -1))
        ...  # TODO: values of all n x |V| candidates, keep the best B, move finished ones aside
    ...  # TODO: return the best hypothesis by value / length

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
@torch.no_grad()
def beam_search(model, source, B, max_len=MAX_OUT):
    """Beam search with length normalization for one source string. Returns (output, log-prob)."""
    model.eval()
    src, src_len, _ = encode_batch([{"src": source, "tgt": ""}])
    src = src.to(device)
    h, h_S = model.encode(src, src_len)  # h: (1, S, d_h)
    mask = src != PAD  # (1, S)
    alive = [([], 0.0)]  # hypotheses: (token ids so far, log-probability)
    state = h_S.unsqueeze(0)  # (1, n_alive, d_h): one decoder state per live hypothesis
    finished = []
    for _ in range(max_len):
        n = len(alive)
        y_prev = torch.tensor([ids[-1] if ids else BOS for ids, _ in alive], device=device)
        log_p, state, _ = decoder_step(model, y_prev, state, h.expand(n, -1, -1), mask.expand(n, -1))
        values = torch.tensor([v for _, v in alive], device=device).unsqueeze(1) + log_p  # (n, |V|)
        top = values.flatten().topk(B)
        next_alive, rows = [], []
        for value, index in zip(top.values.tolist(), top.indices.tolist()):
            row, token = divmod(index, log_p.size(1))  # which hypothesis, which next token
            hypothesis = (alive[row][0] + [token], value)
            if token == EOS:
                finished.append(hypothesis)
            else:
                next_alive.append(hypothesis)
                rows.append(row)
        if len(finished) >= B or not next_alive:
            break
        alive, state = next_alive, state[:, rows]  # keep the decoder state of each survivor
    pool = finished or alive
    ids, value = max(pool, key=lambda hyp: hyp[1] / len(hyp[0]))  # length normalization
    return ids_to_string(ids), value

<details>
<summary>Why this solution works</summary>

Adding the current value to every row of `log_p` gives the value of all $n \times |V|$ one-token extensions; `topk` over the flattened matrix keeps the best $B$, and `divmod` recovers which hypothesis and which token each came from. The decoder state must follow its hypothesis, so `state[:, rows]` reorders and duplicates states to match the survivors. Finished hypotheses leave the beam, which therefore shrinks. With $B = 1$ the single candidate kept at each step is the arg-max token, so the result is exactly greedy decoding.

Dividing by length corrects the bias toward short outputs: every token adds a negative log-probability. Here all outputs for a given source have the same length if the format is right, so normalization rarely changes the winner.

</details>

In [ ]:
# Checkpoint 6 (stretch)
workshop.checkpoint(6)
started = time.time()
subset = [ex for K in (1, 2, 3, 4) for ex in [e for e in test_set if e["K"] == K][:50]]  # 200 sources
greedy_sub, _ = greedy_decode(model_dot, subset)
beam_1 = [beam_search(model_dot, ex["src"], 1)[0] for ex in subset]
assert beam_1 == greedy_sub, "B = 1 must reproduce greedy decoding exactly"

print("Dot-product attention model, 200 test sources (50 per K)\n")
print(" B  exact match  mean log-probability")
for B in (1, 3, 5):
    results = [beam_search(model_dot, ex["src"], B) for ex in subset]
    em = exact_match([r[0] for r in results], [ex["tgt"] for ex in subset])
    print(f"{B:>2}  {em:>11.3f}  {np.mean([r[1] for r in results]):>20.4f}")
print(f"\n({time.time() - started:.0f} s) Checkpoint 6 passed")

**Explain.** A wider beam can find outputs with a higher mean log-probability, because it searches more. Whether it does depends on the model. This one is confident and right (its mean log-probability is close to 0), so greedy decoding already finds the best output, and $B = 3$ and $B = 5$ give the same exact match and the same mean log-probability. Beam search earns its cost when the model is uncertain, as in translation. A larger beam is not guaranteed to score higher on the task metric (briefing, Section 7, "No guarantee"). It also costs about $B$ times as much as greedy decoding.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [5 · The transformer](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/05-transformer-from-scratch.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT